In [1]:
# Importing Libraries 

import pandas as pd
from zipfile import ZipFile

In [2]:
#Extracting Data

with ZipFile("Gaming Dataset.zip") as z:
    ab_test = pd.read_csv(z.open("ab_test.csv"), sep= ";")
    auth_data = pd.read_csv(z.open("auth_data.csv"), sep= ";")
    reg_data = pd.read_csv(z.open("reg_data.csv"), sep= ";")

auth_data["date"] = pd.to_datetime(auth_data["auth_ts"], unit= "s").dt.floor("D")
reg_data["date"] = pd.to_datetime(reg_data["reg_ts"], unit= "s").dt.floor("D")

auth_data = auth_data.drop("auth_ts", axis=1)
reg_data = reg_data.drop("reg_ts", axis=1)

In [3]:
# Data Structure Check

print(ab_test.head())
print(auth_data.head())
print(reg_data.head())

   user_id  revenue testgroup
0        1        0         b
1        2        0         a
2        3        0         a
3        4        0         b
4        5        0         b
   uid       date
0    1 1998-11-18
1    2 1999-07-22
2    2 1999-07-25
3    2 1999-07-31
4    2 1999-08-05
   uid       date
0    1 1998-11-18
1    2 1999-07-22
2    3 2000-01-13
3    4 2000-05-28
4    5 2000-09-16


In [4]:
# Main info about datasets

print(ab_test.info())
print(auth_data.info())
print(reg_data.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 404770 entries, 0 to 404769
Data columns (total 3 columns):
 #   Column     Non-Null Count   Dtype 
---  ------     --------------   ----- 
 0   user_id    404770 non-null  int64 
 1   revenue    404770 non-null  int64 
 2   testgroup  404770 non-null  object
dtypes: int64(2), object(1)
memory usage: 9.3+ MB
None
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9601013 entries, 0 to 9601012
Data columns (total 2 columns):
 #   Column  Dtype         
---  ------  -----         
 0   uid     int64         
 1   date    datetime64[ns]
dtypes: datetime64[ns](1), int64(1)
memory usage: 146.5 MB
None
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 2 columns):
 #   Column  Non-Null Count    Dtype         
---  ------  --------------    -----         
 0   uid     1000000 non-null  int64         
 1   date    1000000 non-null  datetime64[ns]
dtypes: datetime64[ns](1), int64(1)
memory usage: 15

In [5]:
# Customer unique check

print(ab_test["user_id"].nunique())
print(reg_data["uid"].nunique())
print(auth_data["uid"].nunique())

404770
1000000
1000000


In [6]:
# SRM check

ab_test['testgroup'].value_counts(normalize=True)

testgroup
b    0.500697
a    0.499303
Name: proportion, dtype: float64

In [7]:
# Checking activity timestamp

print(auth_data['date'].max())
print(auth_data['date'].min())

2020-09-23 00:00:00
1998-11-18 00:00:00


In [8]:
# Determining the analysis period

testing_date = auth_data['date'].max() - pd.Timedelta(days=30)
auth_data = auth_data[auth_data['date'] >= testing_date]
reg_data = reg_data[reg_data['date'] >= testing_date]

In [9]:
# Dataframe aggregation

last_df = pd.merge(ab_test, auth_data, left_on="user_id", right_on="uid", how="inner")
last_df["is_payer"]  = (last_df["revenue"] > 0).astype(int)
last_df = last_df.groupby("user_id").agg({
    "revenue":"sum",
    "testgroup":"first",
    "date":["min", "max", "count"],
    "is_payer": "max"
}).reset_index()

In [10]:
last_df.columns = [
    "user_id",
    "revenue",
    "group",
    "first_date",
    "last_date",
    "logins",
    "is_payer"
]

In [11]:
last_df.to_csv("ab_test.csv", index=False)

In [12]:
# ARPU calculation

arpu = (
    last_df.groupby("group")["revenue"].sum()
    / last_df.groupby("group")["user_id"].nunique()
)

print(arpu)

group
a    213.465428
b    189.725084
dtype: float64


In [13]:
# Conversion rate calcualtion

cr = last_df.groupby("group")["is_payer"].mean()
print(cr)

group
a    0.011432
b    0.008278
Name: is_payer, dtype: float64


In [14]:
# ARPPU calculation

revenue = last_df.groupby("group")["revenue"].sum()
paying = last_df[last_df["is_payer"] > 0].groupby("group")["user_id"].nunique()

arppu = revenue.div(paying).fillna(0)
print(arppu)

group
a    18672.067308
b    22919.289474
dtype: float64


In [15]:
# Total Revenue calculation

total_revenue = last_df.groupby("group")["revenue"].sum()
print(total_revenue)

group
a    1941895
b    1741866
Name: revenue, dtype: int64


In [21]:
# z-test for conversion

import numpy as np
from statsmodels.stats.proportion import proportions_ztest

success = last_df.groupby("group")["is_payer"].sum()

nobs = last_df.groupby("group")["user_id"].count()

stat, pval = proportions_ztest(success, nobs)

print(stat, pval)

2.1593358783974157 0.03082411857578831
